In [3]:
pip install sentence-transformers numpy

     |████████████████████████████████| 255 kB 3.1 MB/s eta 0:00:01
     |████████████████████████████████| 566 kB 17.8 MB/s eta 0:00:01
     |████████████████████████████████| 34.5 MB 12.2 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 19.1 MB/s eta 0:00:01
     |████████████████████████████████| 10.0 MB 13.2 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 6.4 MB/s  eta 0:00:01
     |████████████████████████████████| 797.1 MB 28 kB/s s eta 0:00:01 |█████                           | 122.8 MB 8.2 MB/s eta 0:01:23     |█████▊                          | 141.6 MB 16.3 MB/s eta 0:00:419 MB 2.9 MB/s eta 0:01:39     |███████████████████████▌        | 586.6 MB 9.6 MB/s eta 0:00:22
     |████████████████████████████████| 4.5 MB 4.9 MB/s eta 0:00:01
     |████████████████████████████████| 193 kB 10.0 MB/s eta 0:00:01
     |████████████████████████████████| 301 kB 12.0 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 8.7 MB/s eta 0:00:01
     |█████

In [2]:


import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ── Your documents ────────────────────────────────────────
documents = [
    "Refunds are accepted within 30 days of purchase.",
    "To request a refund, email support@company.com.",
    "Products must be unused and in original packaging.",
    "Shipping costs are non-refundable.",
]

# ── Embed all docs (done once) ────────────────────────────
doc_embeddings = embedder.encode(documents)  # shape: (4, 384)

# ── Cosine similarity helper ──────────────────────────────
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# ── Retrieve top-K ────────────────────────────────────────
def retrieve(query, top_k=2):
    query_embedding = embedder.encode([query])[0]
    scores = [cosine_similarity(query_embedding, doc_emb)
              for doc_emb in doc_embeddings]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [documents[i] for i in top_indices]

# ── Run it ────────────────────────────────────────────────
query = "Is delivery cost also refundable"
chunks = retrieve(query, top_k=2)

context = "\n".join(chunks)
prompt = f"""Answer using ONLY the context below.

Context:
{context}

Question: {query}
Answer:"""

print(prompt)
# → Pass this to any LLM API

Answer using ONLY the context below.

Context:
Shipping costs are non-refundable.
To request a refund, email support@company.com.

Question: Is delivery cost also refundable
Answer:
